###Run Shared Libraries

In [0]:
%run ../Misc/sharedlibraries

In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimcurrency"

###Read Bronze tables

In [0]:
currencyDf = spark.table("bronze.currency")

In [0]:
currencyDf.printSchema()

###Build Dimension/Fact table

In [0]:
dimcurrencyDf = currencyDf.filter(currencyDf.RecordId.isNotNull()
    ).select(
       currencyDf.CurrencyId,
       F.trim(currencyDf.Code).alias("CurrencyCode"),
       F.when(currencyDf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(currencyDf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
       F.from_utc_timestamp(currencyDf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
       F.trim(currencyDf.Country).alias("Country"),
       F.trim(currencyDf.CurrencyName).alias("CurrencyName"),
       currencyDf.RecordId.alias("CurrencyRecordId")
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("CurrencyHashKey", F.xxhash64("CurrencyRecordId")
    )

display(dimcurrencyDf)

###Final dataframe

In [0]:
df_final = dimcurrencyDf

## Write to Silver Schema

In [0]:
saveDeltaTableToCatalog(df_final,"silver",Entity)